# E5 — Normal-aware loss full-stage (UNet 224×224)

Attach two Inputs before saving this notebook version: the preprocessed BTXRD Dataset and the Output Dataset/version produced by E4. Internet must be enabled to clone the branch. The notebook reads E4's validation ranking, then runs BCE+Dice and at most two eligible challengers for five seeds.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import torch
import yaml

REPO_URL = 'https://github.com/lehngoc/BTXRD-LViT.git'
BRANCH = 'model/loss-ablation-normal-fp'
REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')
WORK_ROOT = Path('/kaggle/working/experiments/loss_ablation')

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle Notebook Settings.'
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'PyYAML'], check=True)

In [ ]:
def find_data_root() -> Path:
    suffix = 'data/exports/btxrd_preprocessed/train.csv'
    for csv_path in Path('/kaggle/input').rglob('train.csv'):
        if csv_path.as_posix().endswith(suffix):
            return csv_path.parents[3]
    raise FileNotFoundError('Missing attached BTXRD preprocessed Dataset.')

def find_screening_summary() -> Path:
    matches = list(Path('/kaggle/input').rglob('screening_loss_ablation_summary.json'))
    if not matches:
        raise FileNotFoundError('Attach the Output Dataset/version from E4 screening.')
    return matches[0]

DATA_ROOT = find_data_root()
screening_summary_path = find_screening_summary()
screening_root = screening_summary_path.parent
WORK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(screening_root, WORK_ROOT / 'screening', dirs_exist_ok=True)
screening = json.loads(screening_summary_path.read_text(encoding='utf-8'))
selected_challengers = screening['ranked_challengers'][:2]
full_losses = ['bce_dice_05'] + selected_challengers
print('Full-stage losses:', full_losses)

In [ ]:
RUNTIME_DIR = Path('/kaggle/working/runtime_configs/full')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
source_by_loss = {}
for source in (REPO_ROOT / 'configs/loss_ablation').glob('*.yaml'):
    cfg = yaml.safe_load(source.read_text(encoding='utf-8'))
    source_by_loss[cfg['experiment']['loss_id']] = source

def make_runtime_config(loss_id: str, seed: int) -> Path:
    cfg = yaml.safe_load(source_by_loss[loss_id].read_text(encoding='utf-8'))
    cfg['data']['root_dir'] = str(DATA_ROOT)
    cfg['training']['device'] = 'cuda'
    cfg['training']['num_workers'] = 2
    cfg['training']['epochs'] = 200
    cfg['training']['seed'] = seed
    cfg['training']['output_dir'] = str(WORK_ROOT / 'full' / loss_id / f'seed{seed}')
    destination = RUNTIME_DIR / loss_id / f'seed{seed}.yaml'
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    return destination

for loss_id in full_losses:
    for seed in [42, 52, 62, 72, 82]:
        runtime_config = make_runtime_config(loss_id, seed)
        print(f'===== full: {loss_id}, seed={seed} =====')
        subprocess.run([sys.executable, '-m', 'src.training.train_unet', '--config', str(runtime_config)], cwd=REPO_ROOT, check=True)

In [ ]:
full_root = WORK_ROOT / 'full'
subprocess.run([
    sys.executable, '-m', 'src.training.aggregate_loss_ablation',
    '--runs-root', str(full_root), '--stage', 'full',
], cwd=REPO_ROOT, check=True)

summary = json.loads((full_root / 'full_loss_ablation_summary.json').read_text(encoding='utf-8'))
print(json.dumps({
    'best_challenger': summary['best_challenger'],
    'validation_winner_overall': summary['validation_winner_overall'],
}, indent=2))

archive = shutil.make_archive('/kaggle/working/loss_ablation_full_artifacts', 'gztar',
                              root_dir='/kaggle/working', base_dir='experiments/loss_ablation')
print('Saved archive:', archive)